# RAG 기반 인재 추천

프로젝트 요구사항 텍스트를 입력받아 LLM이 포지션별로 파싱하고, 벡터 검색으로 적합한 엔지니어를 추천합니다.

In [27]:
from _bootstrap import setup_project_path

setup_project_path()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


WindowsPath('C:/Users/min/Desktop/project-v2/retrieval-lab')

In [28]:
import hashlib
import json
from pathlib import Path

from embedding_retrieval.config import RetrievalConfig
from embedding_retrieval.factory import create_llm, create_retrieval_pipeline
from embedding_retrieval.scenarios.sample_data import SAMPLE_ENGINEERS

config = RetrievalConfig.from_env().with_overrides(chunk_size=2000, chunk_overlap=0, top_k=3)
ingest_service, search_service = create_retrieval_pipeline(config)
llm = create_llm(config)

# 데이터 해시 비교 → 변경 시에만 재임베딩
HASH_FILE = Path("../.ingest_hash")
current_hash = hashlib.md5(json.dumps(SAMPLE_ENGINEERS, ensure_ascii=False, sort_keys=True).encode()).hexdigest()
stored_hash = HASH_FILE.read_text().strip() if HASH_FILE.exists() else ""

if current_hash != stored_hash:
    print("데이터 변경 감지 → 재임베딩 시작...")
    ingest_service.add_texts(
        texts=[item["text"] for item in SAMPLE_ENGINEERS],
        metadatas=[item["metadata"] for item in SAMPLE_ENGINEERS],
    )
    HASH_FILE.write_text(current_hash)
    print(f"완료. 해시: {current_hash}")
else:
    print(f"데이터 변경 없음 → 임베딩 스킵 (해시: {current_hash})")

데이터 변경 감지 → 재임베딩 시작...
완료. 해시: 4ce2104ca0f164263ebeb26f03ffba4f


In [29]:
# 사용자 입력 (프로젝트 요구사항 텍스트)
project_requirement = """
[프로젝트 명]
SK ITSM 이전 서비스

[프로젝트 개요]
기존 운영되던 ITSM 서비스 중 3개의 서비스를 마이그레이션

[총 인원 수]
3명

[요구인력]
 - 포지션: PL / 역할 : 백엔드 개발자 / 등급 : 고급, 특급 / 스킬 : Java, Spring / 인원 1명
 - 포지션 : 개발자 / 역할 : 프론트 / 등급 : 중급 / 스킬 : React / 인원 2명
"""

print(project_requirement)


[프로젝트 명]
SK ITSM 이전 서비스

[프로젝트 개요]
기존 운영되던 ITSM 서비스 중 3개의 서비스를 마이그레이션

[총 인원 수]
3명

[요구인력]
 - 포지션: PL / 역할 : 백엔드 개발자 / 등급 : 고급, 특급 / 스킬 : Java, Spring / 인원 1명
 - 포지션 : 개발자 / 역할 : 프론트 / 등급 : 중급 / 스킬 : React / 인원 2명



In [30]:
import json

# LLM으로 요구사항 파싱 → 포지션별 구조화
parse_prompt = f"""
아래 프로젝트 요구사항에서 포지션별 요구 인력을 JSON 배열로 추출하세요.

등급 변환 규칙:
- 특급 → EXPERT
- 고급 → SENIOR
- 중급 → INTERMEDIATE
- 초급 → JUNIOR

출력 형식 (JSON 배열만 출력, 설명 없이):
[
  {{
    "position": "포지션명",
    "role": "역할",
    "grades": ["SENIOR"],
    "skills": ["Java", "Spring"],
    "count": 1,
    "search_query": "벡터 검색에 사용할 자연어 쿼리"
  }}
]

프로젝트 요구사항:
{project_requirement}
"""

response = llm.invoke(parse_prompt)
raw = response.content.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
positions = json.loads(raw)
positions

ChatGoogleGenerativeAIError: Error calling model 'gemini-2.0-flash-lite' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash-lite\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash-lite\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash-lite\nPlease retry in 56.702315934s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemi

In [ ]:
# 포지션별 벡터 검색 (등급 필터 + 유사도 검색)
results_by_position = {}

for pos in positions:
    grade_filter = " OR ".join([f"grade = '{g}'" for g in pos["grades"]])
    filter_expr = f"status = 'AVAILABLE' AND ({grade_filter})"

    results = search_service.search(
        query=pos["search_query"],
        top_k=pos["count"] + 2,
        filter=filter_expr,
    )
    results_by_position[pos["position"]] = {"requirement": pos, "candidates": results}

# 결과 출력
for position, data in results_by_position.items():
    req = data["requirement"]
    print(f"\n{'='*60}")
    print(f"[{position}] {req['role']} | 등급: {req['grades']} | 스킬: {req['skills']} | {req['count']}명 필요")
    print(f"{'='*60}")
    for r in data["candidates"]:
        print(f"  score: {r.score:.4f} | {r.metadata['engineer_id']} | {r.metadata['grade']} | {r.metadata['engineer_role']}")
        print(f"  {r.document.page_content[:100].strip()}...")
        print()

In [ ]:
# LLM 최종 추천
context_parts = []
for position, data in results_by_position.items():
    req = data["requirement"]
    candidates_text = "\n".join([
        f"- {r.metadata['engineer_id']} (등급: {r.metadata['grade']}, 유사도: {r.score:.4f})\n{r.document.page_content.strip()}"
        for r in data["candidates"]
    ])
    context_parts.append(f"[{position} - {req['role']}]\n{candidates_text}")

context = "\n\n".join(context_parts)

recommend_prompt = f"""
아래 프로젝트 요구사항과 후보 엔지니어 프로필을 보고, 각 포지션에 가장 적합한 엔지니어를 추천하세요.

프로젝트 요구사항:
{project_requirement}

후보 엔지니어 프로필:
{context}

각 포지션별로 최적 후보를 추천하고, 추천 이유를 간략히 설명하세요.
"""

response = llm.invoke(recommend_prompt)
print(response.content)